# Tana (T) — After irrigation (AI) — GA-G vs GA-GPF

This notebook compares GA-G and GA-GPF for after-irrigation worksheets
identified as preferential-flow cases. For each formulation, it evaluates
fully optimised parameters, fixed ψ, and measured Δθ held fixed.

**Input workbook:** `data/Infiltration_T_sat_GA.xlsx`  
**PF worksheets:** 1.1, 2.2, 2.4, 3.4, 3.6, 4.6

The analytical expressions and sheet selection are retained from the source
notebook. Input data are not included.


In [ ]:
from pathlib import Path
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, r2_score

# The notebooks can be run either from the project root or from /notebooks.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PSI_FIXED_CM = -31.63
INITIAL_PARAMS = np.array([0.0001, 0.35, PSI_FIXED_CM], dtype=float)
PSI_BOUNDS = (-156.5, -4.08)
KFS_BOUNDS = (0.00001, 1.0)

INPUT_FILE = DATA_DIR / "Infiltration_T_sat_GA.xlsx"
OUTPUT_STEM = "T_AI_GA_GPF"
PF_SHEETS = ['1.1', '2.2', '2.4', '3.4', '3.6', '4.6']
DELTA_THETA_BOUNDS = (0.18, 0.52)
INITIAL_PARAMS_FIXED_DELTA = np.array([0.0001, PSI_FIXED_CM], dtype=float)
FIXED_DELTA_BOUNDS = [KFS_BOUNDS, PSI_BOUNDS]

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input workbook not found: {INPUT_FILE}\n"
        "Place the requested workbook in the project data/ directory."
    )


In [ ]:
def green_ampt_time_normal(I, Kfs, delta_theta, psi, H0):
    """Green-Ampt time prediction for the standard GA-G formulation."""
    I = np.asarray(I, dtype=float)
    denominator = delta_theta * (H0 - psi)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        return (1.0 / (Kfs * (1.0 - delta_theta))) * (
            I
            - (denominator / (1.0 - delta_theta))
            * np.log1p((I * (1.0 - delta_theta)) / denominator)
        )


def green_ampt_time_pf(I, Kfs, delta_theta, psi, H0, I0):
    """Green-Ampt time prediction for the preferential-flow GA-GPF formulation."""
    I = np.asarray(I, dtype=float)
    storage = (delta_theta * (H0 - psi)) / (1.0 - delta_theta) + I0
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        return (1.0 / (Kfs * (1.0 - delta_theta))) * (
            I - I0 - storage * np.log1p((I - I0) / storage)
        )


def objective_normal(params, I_obs, t_obs, H0):
    Kfs, delta_theta, psi = params
    t_pred = green_ampt_time_normal(I_obs, Kfs, delta_theta, psi, H0)
    if not np.all(np.isfinite(t_pred)):
        return np.inf
    return float(np.sum((t_obs - t_pred) ** 2))


def objective_pf(params, I_obs, t_obs, H0, I0):
    # This preserves the objective used in the original notebook.
    Kfs, delta_theta, psi = params
    t_pred = green_ampt_time_pf(I_obs, Kfs, delta_theta, psi, H0, I0)
    if not np.all(np.isfinite(t_pred)):
        return np.inf
    return float(np.sum((t_obs - t_pred) ** 2))


def objective_fixed_delta_theta(params, I_obs, t_obs, H0, delta_theta):
    Kfs, psi = params
    t_pred = green_ampt_time_normal(I_obs, Kfs, delta_theta, psi, H0)
    if not np.all(np.isfinite(t_pred)):
        return np.inf
    return float(np.sum((t_obs - t_pred) ** 2))


def objective_fixed_delta_theta_pf(params, I_obs, t_obs, H0, delta_theta, I0):
    Kfs, psi = params
    t_pred = green_ampt_time_pf(I_obs, Kfs, delta_theta, psi, H0, I0)
    if not np.all(np.isfinite(t_pred)):
        return np.inf
    return float(np.sum((t_obs - t_pred) ** 2))


def rmse(y_obs, y_pred):
    return float(np.sqrt(mean_squared_error(y_obs, y_pred)))


def r_squared(y_obs, y_pred):
    return float(r2_score(y_obs, y_pred))


def kfs_from_last_point(I_last, t_last, delta_theta, psi, H0):
    """Original fixed-psi Kfs calculation retained from the source notebook."""
    denominator = delta_theta * (H0 - psi)
    return (1.0 / (t_last * (1.0 - delta_theta))) * (
        I_last
        - (denominator / (1.0 - delta_theta))
        * np.log1p((I_last * (1.0 - delta_theta)) / denominator)
    )


def load_sheet(xls, sheet_name, required_columns):
    df = pd.read_excel(xls, sheet_name=sheet_name)
    missing = [column for column in required_columns if column not in df.columns]
    if missing:
        raise KeyError(
            f"Worksheet {sheet_name!r} is missing required columns: {missing}"
        )

    clean = df.loc[:, required_columns].copy()
    for column in required_columns:
        clean[column] = pd.to_numeric(clean[column], errors="coerce")
    clean = clean.dropna()

    if len(clean) < 3:
        raise ValueError(
            f"Worksheet {sheet_name!r} has fewer than three complete observations."
        )
    return clean


In [ ]:
xls = pd.ExcelFile(INPUT_FILE)
missing_sheets = [sheet for sheet in PF_SHEETS if sheet not in xls.sheet_names]
if missing_sheets:
    warnings.warn(f"PF worksheets not found and skipped: {missing_sheets}")
sheets = [sheet for sheet in PF_SHEETS if sheet in xls.sheet_names]

if not sheets:
    raise ValueError("None of the requested PF worksheets exists in the workbook.")

required_columns = [
    "Ia",
    "Start Time (min)",
    "Start Level (cm)",
    "Delta Theta",
]

n_rows = math.ceil(len(sheets) / 2)
fig, axes = plt.subplots(n_rows, 4, figsize=(30, n_rows * 8), squeeze=False)
result_rows = []

for i, sheet_name in enumerate(sheets):
    try:
        df = load_sheet(xls, sheet_name, required_columns)
        I_obs = df["Ia"].to_numpy(dtype=float)
        t_obs = df["Start Time (min)"].to_numpy(dtype=float)
        H0 = float(df["Start Level (cm)"].iloc[0])
        delta_theta_measured = float(df["Delta Theta"].iloc[0])

        if len(I_obs) < 2:
            raise ValueError("At least two infiltration observations are required.")
        I0 = float(I_obs[1])

        bounds = [KFS_BOUNDS, DELTA_THETA_BOUNDS, PSI_BOUNDS]

        result_normal = minimize(
            objective_normal,
            INITIAL_PARAMS,
            args=(I_obs, t_obs, H0),
            method="L-BFGS-B",
            bounds=bounds,
        )
        result_pf = minimize(
            objective_pf,
            INITIAL_PARAMS,
            args=(I_obs, t_obs, H0, I0),
            method="L-BFGS-B",
            bounds=bounds,
        )
        result_fixed_delta_normal = minimize(
            objective_fixed_delta_theta,
            INITIAL_PARAMS_FIXED_DELTA,
            args=(I_obs, t_obs, H0, delta_theta_measured),
            method="L-BFGS-B",
            bounds=FIXED_DELTA_BOUNDS,
        )
        result_fixed_delta_pf = minimize(
            objective_fixed_delta_theta_pf,
            INITIAL_PARAMS_FIXED_DELTA,
            args=(I_obs, t_obs, H0, delta_theta_measured, I0),
            method="L-BFGS-B",
            bounds=FIXED_DELTA_BOUNDS,
        )

        optimisation_results = [
            result_normal,
            result_pf,
            result_fixed_delta_normal,
            result_fixed_delta_pf,
        ]
        if not all(result.success for result in optimisation_results):
            messages = [result.message for result in optimisation_results]
            warnings.warn(
                f"At least one optimisation failed for {sheet_name}: {messages}"
            )
            continue

        Kfs_normal, delta_normal, psi_normal = result_normal.x
        t_pred_normal = green_ampt_time_normal(
            I_obs, Kfs_normal, delta_normal, psi_normal, H0
        )
        rmse_normal = rmse(t_obs, t_pred_normal)
        r2_normal = r_squared(t_obs, t_pred_normal)

        Kfs_pf, delta_pf, psi_pf = result_pf.x
        t_pred_pf = np.concatenate(
            (
                [0.0],
                green_ampt_time_pf(
                    I_obs[1:], Kfs_pf, delta_pf, psi_pf, H0, I0
                ),
            )
        )
        rmse_pf = rmse(t_obs, t_pred_pf)
        r2_pf = r_squared(t_obs, t_pred_pf)

        Kfs_fixed_psi_normal = kfs_from_last_point(
            I_obs[-1], t_obs[-1], delta_normal, PSI_FIXED_CM, H0
        )
        t_pred_fixed_psi_normal = green_ampt_time_normal(
            I_obs, Kfs_fixed_psi_normal, delta_normal, PSI_FIXED_CM, H0
        )
        rmse_fixed_psi_normal = rmse(t_obs, t_pred_fixed_psi_normal)
        r2_fixed_psi_normal = r_squared(t_obs, t_pred_fixed_psi_normal)

        Kfs_fixed_psi_pf = kfs_from_last_point(
            I_obs[-1], t_obs[-1], delta_pf, PSI_FIXED_CM, H0
        )
        t_pred_fixed_psi_pf = np.concatenate(
            (
                [0.0],
                green_ampt_time_pf(
                    I_obs[1:],
                    Kfs_fixed_psi_pf,
                    delta_pf,
                    PSI_FIXED_CM,
                    H0,
                    I0,
                ),
            )
        )
        rmse_fixed_psi_pf = rmse(t_obs, t_pred_fixed_psi_pf)
        r2_fixed_psi_pf = r_squared(t_obs, t_pred_fixed_psi_pf)

        Kfs_fixed_delta_normal, psi_fixed_delta_normal = (
            result_fixed_delta_normal.x
        )
        t_pred_fixed_delta_normal = green_ampt_time_normal(
            I_obs,
            Kfs_fixed_delta_normal,
            delta_theta_measured,
            psi_fixed_delta_normal,
            H0,
        )
        rmse_fixed_delta_normal = rmse(t_obs, t_pred_fixed_delta_normal)
        r2_fixed_delta_normal = r_squared(t_obs, t_pred_fixed_delta_normal)

        Kfs_fixed_delta_pf, psi_fixed_delta_pf = result_fixed_delta_pf.x
        t_pred_fixed_delta_pf = np.concatenate(
            (
                [0.0],
                green_ampt_time_pf(
                    I_obs[1:],
                    Kfs_fixed_delta_pf,
                    delta_theta_measured,
                    psi_fixed_delta_pf,
                    H0,
                    I0,
                ),
            )
        )
        rmse_fixed_delta_pf = rmse(t_obs, t_pred_fixed_delta_pf)
        r2_fixed_delta_pf = r_squared(t_obs, t_pred_fixed_delta_pf)

        result_rows.extend(
            [
                {
                    "Sample": sheet_name,
                    "Model": "GA-G",
                    "Parameterisation": "Optimised",
                    "Psi_cm": psi_normal,
                    "Delta_theta": delta_normal,
                    "Kfs_cm_min": Kfs_normal,
                    "I0_cm": I0,
                    "RMSE_min": rmse_normal,
                    "R2": r2_normal,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-G",
                    "Parameterisation": "Fixed psi",
                    "Psi_cm": PSI_FIXED_CM,
                    "Delta_theta": delta_normal,
                    "Kfs_cm_min": Kfs_fixed_psi_normal,
                    "I0_cm": I0,
                    "RMSE_min": rmse_fixed_psi_normal,
                    "R2": r2_fixed_psi_normal,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-G",
                    "Parameterisation": "Fixed measured delta theta",
                    "Psi_cm": psi_fixed_delta_normal,
                    "Delta_theta": delta_theta_measured,
                    "Kfs_cm_min": Kfs_fixed_delta_normal,
                    "I0_cm": I0,
                    "RMSE_min": rmse_fixed_delta_normal,
                    "R2": r2_fixed_delta_normal,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-GPF",
                    "Parameterisation": "Optimised",
                    "Psi_cm": psi_pf,
                    "Delta_theta": delta_pf,
                    "Kfs_cm_min": Kfs_pf,
                    "I0_cm": I0,
                    "RMSE_min": rmse_pf,
                    "R2": r2_pf,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-GPF",
                    "Parameterisation": "Fixed psi",
                    "Psi_cm": PSI_FIXED_CM,
                    "Delta_theta": delta_pf,
                    "Kfs_cm_min": Kfs_fixed_psi_pf,
                    "I0_cm": I0,
                    "RMSE_min": rmse_fixed_psi_pf,
                    "R2": r2_fixed_psi_pf,
                },
                {
                    "Sample": sheet_name,
                    "Model": "GA-GPF",
                    "Parameterisation": "Fixed measured delta theta",
                    "Psi_cm": psi_fixed_delta_pf,
                    "Delta_theta": delta_theta_measured,
                    "Kfs_cm_min": Kfs_fixed_delta_pf,
                    "I0_cm": I0,
                    "RMSE_min": rmse_fixed_delta_pf,
                    "R2": r2_fixed_delta_pf,
                },
            ]
        )

        row = i // 2
        pair = i % 2
        ax_normal = axes[row, pair * 2]
        ax_pf = axes[row, pair * 2 + 1]

        for ax in (ax_normal, ax_pf):
            ax.scatter(t_obs, I_obs, color="black")
            ax.set_xlabel("t (min)")
            ax.set_ylabel("I (cm)")
            ax.grid(True)

        ax_normal.plot(t_pred_normal, I_obs, color="blue", linestyle="-")
        ax_normal.plot(
            t_pred_fixed_psi_normal, I_obs, color="red", linestyle="--"
        )
        ax_normal.plot(
            t_pred_fixed_delta_normal,
            I_obs,
            color="darkgreen",
            linestyle=":",
        )
        ax_normal.set_title(f"{sheet_name} — GA-G")

        ax_pf.plot(t_pred_pf, I_obs, color="blue", linestyle="-")
        ax_pf.plot(t_pred_fixed_psi_pf, I_obs, color="red", linestyle="--")
        ax_pf.plot(
            t_pred_fixed_delta_pf, I_obs, color="darkgreen", linestyle=":"
        )
        ax_pf.set_title(f"{sheet_name} — GA-GPF")

        normal_text = "\n".join(
            [
                "Optimised",
                f"ψ = {psi_normal:.2f} cm",
                f"Δθ = {delta_normal:.2f}",
                f"$K_{{fs}}$ = {Kfs_normal:.4f} cm/min",
                f"RMSE = {rmse_normal:.2f} min",
                f"$R^2$ = {r2_normal:.2f}",
            ]
        )
        normal_fixed_psi_text = "\n".join(
            [
                f"Fixed ψ = {PSI_FIXED_CM:.2f} cm",
                f"$K_{{fs}}$ = {Kfs_fixed_psi_normal:.4f} cm/min",
                f"RMSE = {rmse_fixed_psi_normal:.2f} min",
                f"$R^2$ = {r2_fixed_psi_normal:.2f}",
            ]
        )
        normal_fixed_delta_text = "\n".join(
            [
                "Fixed measured Δθ",
                f"ψ = {psi_fixed_delta_normal:.2f} cm",
                f"Δθ = {delta_theta_measured:.2f}",
                f"$K_{{fs}}$ = {Kfs_fixed_delta_normal:.4f} cm/min",
                f"RMSE = {rmse_fixed_delta_normal:.2f} min",
                f"$R^2$ = {r2_fixed_delta_normal:.2f}",
            ]
        )
        pf_text = "\n".join(
            [
                "Optimised",
                f"ψ = {psi_pf:.2f} cm",
                f"Δθ = {delta_pf:.2f}",
                f"$K_{{fs}}$ = {Kfs_pf:.4f} cm/min",
                f"$I_0$ = {I0:.2f} cm",
                f"RMSE = {rmse_pf:.2f} min",
                f"$R^2$ = {r2_pf:.2f}",
            ]
        )
        pf_fixed_psi_text = "\n".join(
            [
                f"Fixed ψ = {PSI_FIXED_CM:.2f} cm",
                f"$K_{{fs}}$ = {Kfs_fixed_psi_pf:.4f} cm/min",
                f"RMSE = {rmse_fixed_psi_pf:.2f} min",
                f"$R^2$ = {r2_fixed_psi_pf:.2f}",
            ]
        )
        pf_fixed_delta_text = "\n".join(
            [
                "Fixed measured Δθ",
                f"ψ = {psi_fixed_delta_pf:.2f} cm",
                f"Δθ = {delta_theta_measured:.2f}",
                f"$K_{{fs}}$ = {Kfs_fixed_delta_pf:.4f} cm/min",
                f"RMSE = {rmse_fixed_delta_pf:.2f} min",
                f"$R^2$ = {r2_fixed_delta_pf:.2f}",
            ]
        )

        ax_normal.text(
            0.97,
            0.64,
            normal_text,
            transform=ax_normal.transAxes,
            va="center",
            ha="right",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax_normal.text(
            0.97,
            0.28,
            normal_fixed_delta_text,
            transform=ax_normal.transAxes,
            va="center",
            ha="right",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax_normal.text(
            0.97,
            0.03,
            normal_fixed_psi_text,
            transform=ax_normal.transAxes,
            va="bottom",
            ha="right",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )

        ax_pf.text(
            0.97,
            0.64,
            pf_text,
            transform=ax_pf.transAxes,
            va="center",
            ha="right",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax_pf.text(
            0.97,
            0.28,
            pf_fixed_delta_text,
            transform=ax_pf.transAxes,
            va="center",
            ha="right",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )
        ax_pf.text(
            0.97,
            0.03,
            pf_fixed_psi_text,
            transform=ax_pf.transAxes,
            va="bottom",
            ha="right",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "none"},
        )

    except Exception as exc:
        warnings.warn(f"Worksheet {sheet_name!r} was skipped: {exc}")

used_axes = len(sheets) * 2
for ax in axes.ravel()[used_axes:]:
    ax.remove()

handles = [
    plt.Line2D([0], [0], marker="o", color="black", linestyle="None"),
    plt.Line2D([0], [0], color="blue", linestyle="-"),
    plt.Line2D([0], [0], color="red", linestyle="--"),
    plt.Line2D([0], [0], color="darkgreen", linestyle=":"),
]
labels = [
    "Observed data",
    "Optimised model",
    f"Model with ψ = {PSI_FIXED_CM:.2f} cm",
    "Model with measured Δθ",
]
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.05, 1, 1))

plot_path = OUTPUT_DIR / f"{OUTPUT_STEM}_plots.png"
results_path = OUTPUT_DIR / f"{OUTPUT_STEM}_results.csv"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

results_df = pd.DataFrame(result_rows)
results_df.to_csv(results_path, index=False)

print(f"Processed worksheets: {results_df['Sample'].nunique() if not results_df.empty else 0}")
print(f"Results: {results_path}")
print(f"Figure:  {plot_path}")
results_df
